# Augmenting EM-DAT Data with AI Agents and Web Search

This tutorial shows how to enrich EM-DAT records with **disaster storylines**: short, structured narratives describing an event, its severity, drivers, and impacts. It is inspired by [Ronco et al. (2026)](https://doi.org/10.1038/s41597-026-07036-2), who generated storylines for over 3,000 EM-DAT events (2014–2024) by feeding news articles from the Europe Media Monitor to a Large Language Model (LLM).

Here, we replace the news archive with a [Mistral AI agent](https://docs.mistral.ai/studio/agents/introduction) that searches the web by itself and cites its sources. The list of events comes from the [EM-DAT Archive](https://doi.org/10.14428/DVN/I0LTPH).

You will learn how to:

- retrieve an EM-DAT record with pandas;
- create a Mistral agent equipped with a web search tool;
- turn the record into a storyline with linked sources;
- use the storylines published by Ronco et al. and compare them with those of the agent;
- extract structured data from a storyline, such as a severity level and impact figures, and reuse it as JSON or in a summary.

**Requirements**:

1. Python ≥ 3.11 with `pandas`, `openpyxl`, and `mistralai`, which also installs `pydantic`. In a clone of the [tutorial repository](https://github.com/em-dat/python_tutorials), `uv sync` installs them (see its README); otherwise, use `pip install pandas openpyxl mistralai`. This tutorial was tested with pandas 3.0, mistralai 3.2, and pydantic 2.14.
2. A Mistral API key, created in the [Mistral console](https://console.mistral.ai/). API calls, including web searches, are billed to your account.
3. The file `260430_emdat_archive.xlsx`, downloaded from the [EM-DAT Archive](https://doi.org/10.14428/DVN/I0LTPH) into a `data/` folder.

**Note**: LLM outputs can vary from one run to another, and web content changes over time, so your storylines may differ from those shown here.

## Setup

### Import Modules

Let us import the necessary modules and print the versions of the main packages.

In [1]:
import os
import re
from functools import lru_cache
from getpass import getpass
from importlib.metadata import version
from typing import Literal

import pandas as pd
from IPython.display import Markdown, display
from mistralai.client import Mistral
from pydantic import BaseModel, Field

print(f"pandas version: {version('pandas')}")
print(f"mistralai version: {version('mistralai')}")
print(f"pydantic version: {version('pydantic')}")

pandas version: 3.0.6
mistralai version: 3.2.0
pydantic version: 2.14.1


**Note**: Since version 2.0 of the Mistral SDK, the client is imported from `mistralai.client`. Older examples using `from mistralai import Mistral` fail with recent versions.

### Connect to Mistral

Never write your API key in a notebook: it is too easy to share it by mistake. Store it instead in the `MISTRAL_API_KEY` environment variable, in the terminal from which you launch Jupyter:

- macOS/Linux: `export MISTRAL_API_KEY="your-key"`
- Windows PowerShell: `$env:MISTRAL_API_KEY="your-key"`

If you run Jupyter from an IDE, such as PyCharm or VS Code, set the variable in your system settings instead, and restart the IDE so that it reads the new value.

The code below reads the variable with [`os.environ`](https://docs.python.org/3/library/os.html#os.environ) and, if it is missing, asks for the key with [`getpass`](https://docs.python.org/3/library/getpass.html), which hides what you type. If the variable holds an outdated key, the API rejects it with a `401` error ("Invalid API Key") when the agent is created.

In [2]:
api_key = os.environ.get("MISTRAL_API_KEY") or getpass("Mistral API key: ")
client = Mistral(api_key=api_key)

**Note**: Other providers, such as [OpenAI](https://developers.openai.com/api/docs/guides/tools-web-search) and [Anthropic (Claude)](https://platform.claude.com/docs/en/agents-and-tools/tool-use/web-search-tool), offer similar web search tools. With them, the first sections of this tutorial would be much the same: only the package and the client change, while the EM-DAT data and the instructions stay as they are. Creating the agent, calling it, and reading its answer follow their own APIs, described in their documentation.

## Load EM-DAT Data

We load only the columns we need with [`pd.read_excel`](https://pandas.pydata.org/docs/reference/api/pandas.read_excel.html) and use the disaster number `DisNo.` as the index, so that any event can be retrieved with [`.loc`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.loc.html). Column definitions are given in the [EM-DAT documentation](https://doc.emdat.be/docs/data-structure-and-content/emdat-public-table/).

Two details are worth noting:

- Reading the Excel file takes a few seconds. The [`@lru_cache`](https://docs.python.org/3/library/functools.html#functools.lru_cache) decorator keeps the result of `load_emdat` in memory, so the file is read only once.
- Month and day columns contain missing values, so pandas reads them as decimals (e.g., `10.0`). We convert them to the [nullable integer type](https://pandas.pydata.org/docs/user_guide/integer_na.html) `Int64`.

In [3]:
KEEP_COLS = [
    "DisNo.",
    "Disaster Type",
    "Disaster Subtype",
    "Associated Types",
    "Event Name",
    "Country",
    "Subregion",
    "Location",
    "Start Year",
    "Start Month",
    "Start Day",
    "End Year",
    "End Month",
    "End Day",
    "Total Deaths",
    "Total Affected",
    "Total Damage ('000 US$)",
]
DATE_COLS = ["Start Year", "Start Month", "Start Day", "End Year", "End Month", "End Day"]


@lru_cache
def load_emdat(emdat_path: str = "data/260430_emdat_archive.xlsx") -> pd.DataFrame:
    emdat = pd.read_excel(emdat_path, index_col="DisNo.", usecols=KEEP_COLS)
    emdat[DATE_COLS] = emdat[DATE_COLS].astype("Int64")
    return emdat


def load_by_disno(disaster_number: str) -> pd.Series:
    emdat = load_emdat()
    return emdat.loc[disaster_number]

Let us retrieve the record of the October 2024 floods in Spain, which hit the Valencia region the hardest. With over 220 deaths, they were the deadliest floods in the country since 1973 ([Charalampous et al., 2025](https://doi.org/10.3389/ijph.2025.1608236)).

In [4]:
disaster = load_by_disno("2024-0796-ESP")
disaster

Disaster Type                                                          Flood
Disaster Subtype                                             Flood (General)
Event Name                                                               NaN
Country                                                                Spain
Subregion                                                    Southern Europe
Location                   Alaquàs, Albal, Aldaya, Alfafar, Algemesí, Ben...
Associated Types                                            Lightening|Storm
Start Year                                                              2024
Start Month                                                               10
Start Day                                                                 27
End Year                                                                2024
End Month                                                                 11
End Day                                                                    4

## Build a Storyline Agent

An *agent* bundles a model, instructions, and tools. Here, the tool is Mistral's built-in [web search](https://docs.mistral.ai/studio/agents/agent-tools/websearch): when the model needs information, it searches the web, reads the results, and answers with references to the pages it used. Mistral runs the searches on its side, so we have nothing to install.

### Write the Instructions

We reuse the storyline template of Ronco et al. (2026), stored in a dictionary that maps each section to a short description. For simplicity, we only use its first four sections, leaving out the last three on multi-hazard risks, best practices, and recovery recommendations. The instructions ask the agent to search, to rely only on what it finds, and to write `unknown` rather than guess.

In [5]:
SECTIONS = {
    "Key information": "Quick summary with location and date.",
    "Severity": "Low, Medium, High.",
    "Key drivers": "Main causes of the disaster.",
    "Main impacts, exposure, and vulnerability": "Economic damage, people affected, fatalities, effects on communities and infrastructure.",
}

template = "\n".join(f"### {section}\n[{description}]" for section, description in SECTIONS.items())

INSTRUCTIONS = f"""You are an assistant tasked with providing concise and factual updates on disaster events.
For the event described by the user:
1. Search the web for news reports and official sources about this specific event.
2. Complete the factsheet below using only the information you found, and cite your sources as Markdown links: [title](url).
3. If a piece of information is uncertain or not found, write 'unknown'. Do not add assumptions.

Return only the factsheet. Keep the headings exactly as written and replace each bracketed description with 2 to 5 sentences.

{template}"""

print(INSTRUCTIONS)

You are an assistant tasked with providing concise and factual updates on disaster events.
For the event described by the user:
1. Search the web for news reports and official sources about this specific event.
2. Complete the factsheet below using only the information you found, and cite your sources as Markdown links: [title](url).
3. If a piece of information is uncertain or not found, write 'unknown'. Do not add assumptions.

Return only the factsheet. Keep the headings exactly as written and replace each bracketed description with 2 to 5 sentences.

### Key information
[Quick summary with location and date.]
### Severity
[Low, Medium, High.]
### Key drivers
[Main causes of the disaster.]
### Main impacts, exposure, and vulnerability
[Economic damage, people affected, fatalities, effects on communities and infrastructure.]


### Create the Agent

We create the agent with [`client.beta.agents.create`](https://docs.mistral.ai/studio/agents/agents-api). Two settings make the results easier to reproduce, a good practice in research:

- a named model version, `mistral-large-4`, rather than an alias like `mistral-large-latest`, which switches to newer models over time (versions are listed on the [models page](https://docs.mistral.ai/getting-started/models));
- a `temperature` of 0, so that the model picks the most likely words instead of sampling among them, which makes its answers more consistent from one run to another.

In [6]:
agent = client.beta.agents.create(
    model="mistral-large-4",
    name="EM-DAT storyline agent",
    instructions=INSTRUCTIONS,
    tools=[{"type": "web_search"}],
    completion_args={"temperature": 0, "top_p": 1},  # Mistral requires top_p=1 when temperature=0
)
print(f"Agent created: {agent.id}")

Agent created: ag_01a13eba2e1b778682f2984c01a8a306


**Note**: The agent is saved in your Mistral workspace and can be reused in later sessions through its `id`. We delete it at the end of this tutorial.

## Generate a Storyline

### Describe the Event

The agent needs to know which event to document. Like Ronco et al., we describe it with its type, country, and dates from EM-DAT, to which we add the event name and location when available. We leave out the impact figures (deaths, affected people, damage) on purpose, and do not mention EM-DAT, to encourage the agent to find its own sources rather than look up EM-DAT's figures. We can then compare its storyline with EM-DAT.

[`.dropna()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.dropna.html) removes empty fields, such as a missing event name.

In [7]:
EVENT_FIELDS = [
    "Disaster Type",
    "Disaster Subtype",
    "Associated Types",
    "Event Name",
    "Country",
    "Location",
    "Start Year",
    "Start Month",
    "Start Day",
    "End Year",
    "End Month",
    "End Day",
]


def describe_event(disaster: pd.Series) -> str:
    fields = disaster[EVENT_FIELDS].dropna()
    lines = [f"- {name}: {value}" for name, value in fields.items()]
    return "Disaster event:\n" + "\n".join(lines)


print(describe_event(disaster))

Disaster event:
- Disaster Type: Flood
- Disaster Subtype: Flood (General)
- Associated Types: Lightening|Storm
- Country: Spain
- Location: Alaquàs, Albal, Aldaya, Alfafar, Algemesí, Benerússer, Catarroja, Llocnou de la Corona, Massanassa, Paiporta, Picanya, Sedaví, Utiel and La Torre municipalties (Valencia);, Letur, lbacete provinces; Castilla-La Mancha communities, Andalusia, Murcia, and the Balearic Islands, Catalonia
- Start Year: 2024
- Start Month: 10
- Start Day: 27
- End Year: 2024
- End Month: 11
- End Day: 4


### Run the Agent

[`client.beta.conversations.start`](https://docs.mistral.ai/studio/agents/agents-api) sends the description to the agent and returns its answer. This can take a minute or more, depending on how many searches the agent runs.

In [8]:
response = client.beta.conversations.start(agent_id=agent.id, inputs=describe_event(disaster))
response.usage

ConversationUsageInfo(prompt_tokens=1090, completion_tokens=2872, total_tokens=78184, connector_tokens=74222, connectors={'web_search': 8})

The `usage` attribute shows what the call consumed: `connectors` counts the web searches, and `connector_tokens` the search results read by the model, which usually make up most of the tokens.

The response holds a list of `outputs`: `tool.execution` entries for the web searches, and `message.output` entries for the model's messages, the last one being the final answer. Let us see what the agent searched for:

In [9]:
for output in response.outputs:
    if output.type == "tool.execution":
        print(output.arguments)

{"query": "Spain Valencia flood October 2024 DANA Alaquàs Albal Aldaya Alfafar Paiporta death toll damage", "limit": 15}
{"query": "Spain flood October 29 2024 Valencia Castilla-La Mancha Andalusia Murcia Balearic Islands Catalonia casualties economic damage", "limit": 15}
{"query": "DANA flood Spain 2024 official death toll EM-DAT insurance damage Valencia region November 2024", "limit": 15}
{"query": "Spain DANA flood October 2024 economic damage billions euros insurance claims agriculture infrastructure destroyed homes", "limit": 15}
{"query": "Valencia flood 2024 number of people affected displaced evacuated homes destroyed infrastructure damage roads bridges rail", "limit": 15}
{"query": "EM-DAT Spain flood 2024 Valencia disaster number affected deaths October November official statistics", "limit": 15}
{"query": "Spain DANA flood October 2024 EM-DAT disaster number total affected people official statistics Valencia", "limit": 10}
{"query": "\"2024 Spanish floods\" Wikipedia death

### Read the Answer and Its Sources

The final answer is split into *chunks*. As `mistral-large-4` is a reasoning model, `thinking` chunks hold its reasoning, which we skip, while `text` chunks hold the storyline. As requested, the agent cites its sources as Markdown links within the text. Some models return them as separate `tool_reference` chunks instead (see [Citations & References](https://docs.mistral.ai/studio/conversations/citations)), so the function below turns these into links too.

In [10]:
def parse_answer(response) -> str:
    """Return the agent's final answer as Markdown text, with sources as links."""
    message = response.outputs[-1]  # the final answer comes last
    if isinstance(message.content, str):
        return message.content

    text = ""
    for chunk in message.content:
        if chunk.type == "text":
            text += chunk.text
        elif chunk.type == "tool_reference":
            text += f" [{chunk.title}]({chunk.url})"
    return text


text = parse_answer(response)
display(Markdown(text))

### Key information
Between 27 October and 4 November 2024, catastrophic flash floods caused by a DANA (Depresión Aislada en Niveles Altos, or "cold drop") weather phenomenon devastated eastern and southern Spain. The hardest-hit area was the Valencian Community, particularly the Horta Sud municipalities south of Valencia city (including Paiporta, Catarroja, Aldaia, Alfafar, Alaquàs, Albal, Algemesí, Benetússer, Massanassa, Picanya, Sedaví, and Llocnou de la Corona), with additional severe impacts in Castilla-La Mancha (Letur, Albacete), Andalusia, Murcia, the Balearic Islands, and Catalonia. The event peaked on 29 October 2024, when some areas received more than a year's worth of rainfall in just eight hours. [Wikipedia - 2024 Spanish floods](https://en.wikipedia.org/wiki/2024_Spanish_floods) [WMO](https://wmo.int/media/news/devastating-rainfall-hits-spain-yet-another-flood-related-disaster) [BBC News](https://www.bbc.com/news/articles/cx2dp775nd1o)

### Severity
High. This was one of the deadliest natural disasters in Spanish and European history, and the worst floods Spain had seen in decades. The event caused 229-233 fatalities (with the most recent judicial count reaching 233 as of September 2026), affected over 300,000 people, and displaced approximately 15,000 residents. It accounted for 70% of all flood-related deaths in Europe in 2024. [Wikipedia - 2024 Spanish floods](https://en.wikipedia.org/wiki/2024_Spanish_floods) [BBC News](https://www.bbc.com/news/articles/cx2dp775nd1o) [El País English](https://english.elpais.com/climate/2025-04-15/70-of-flood-deaths-in-europe-in-2024-occurred-in-the-valencia-floodplain.html) [The Olive Press](https://www.theolivepress.es/spain-news/2026/09/29/valencia-2024-flood-disaster-death-toll-goes-up-to-233-people/)

### Key drivers
The disaster was triggered by a DANA (Depresión Aislada en Niveles Altos), a meteorological phenomenon where a pocket of cold air detaches from the jet stream and collides with warm, moisture-laden air over the overheated Mediterranean Sea. On 29 October 2024, this system delivered record-breaking rainfall: the Chiva weather station recorded 491 liters per square meter in eight hours (equivalent to a year's rainfall), while Turís received 771 liters per square meter in 24 hours, including 187 liters in a single hour—the most intense rainfall ever recorded in Spain. Climate change attribution studies indicate that human-induced warming made the rainfall intensity approximately twice as likely. Additional contributing factors included rivers overflowing (Turia, Júcar, Magro, and the Rambla del Poyo ravine), drought-hardened soils from previous heat waves that reduced water absorption, and urban expansion into flood-prone areas that lacked adequate drainage infrastructure. [Wikipedia - 2024 Spanish floods](https://en.wikipedia.org/wiki/2024_Spanish_floods) [WMO](https://wmo.int/media/news/devastating-rainfall-hits-spain-yet-another-flood-related-disaster) [Guy Carpenter Post-Event Report](https://www.guycarp.com/insights/2024/11/october-2024-dana-floods-spain.html) [PreventionWeb](https://www.preventionweb.net/news/study-concludes-climate-change-intensified-flooding-caused-dana-valencia-2024) [Euro Weekly News](https://euroweeklynews.com/2025/10/28/one-year-after-the-dana-floods-229-lives-lost-billions-in-damages-and-a-region-still-in-shock/)

### Main impacts, exposure, and vulnerability
The floods caused catastrophic human and economic losses. The death toll reached 229-233 people, with 224-229 in Valencia province, seven in Castilla-La Mancha (including six in Letur), and one in Andalusia; nearly half of the victims were over 70 years old. Over 300,000 people were affected across 75-80 municipalities, with approximately 15,000 residents displaced. Economic damage estimates range from €10.7 billion (Munich Re) to €17.8 billion in direct damages, with total reconstruction costs potentially exceeding €29 billion. The Insurance Compensation Consortium (CCS) recorded at least 116,000-200,000 insurance claims totaling at least €3.5 billion. Infrastructure damage was extensive: 1,450 km of roads, 566 km of railway, 380 bridges, and 2,755 km of forest tracks were damaged or destroyed. Approximately 75,000 homes were flooded, 1,800 businesses were destroyed and 4,500 damaged, and over 100,000-130,000 vehicles were damaged or destroyed. The agricultural sector suffered losses exceeding €1 billion, with 20,000 hectares of farmland affected. Over 100,000 homes lost electricity, and essential services including water supply, sanitation, schools, and healthcare facilities were disrupted. The Spanish government approved a €10.6 billion recovery package, later supplemented by an additional €3.76 billion. [Wikipedia - 2024 Spanish floods](https://en.wikipedia.org/wiki/2024_Spanish_floods) [JBA Risk Management](https://www.jbarisk.com/knowledge-hub/event-response/spain-floods-october-2024/) [Guy Carpenter](https://www.guycarp.com/insights/2024/11/october-2024-dana-floods-spain.html) [Euro Weekly News](https://euroweeklynews.com/2025/10/28/one-year-after-the-dana-floods-229-lives-lost-billions-in-damages-and-a-region-still-in-shock/) [International Journal for Equity in Health](https://equityhealthj.biomedcentral.com/articles/10.1186/s12939-025-02435-0) [Consorcio de Compensación de Seguros](https://consorsegurosdigital.com/en/numero-23/sumario/contributions/valencia_floods/) [Zurich Insurance](https://www.zurich.com/commercial-insurance/sustainability-and-insights/commercial-insurance-risk-insights/valencia-floods-lessons-in-claims-and-recovery-one-year-on)

[`Markdown`](https://ipython.readthedocs.io/en/stable/api/generated/IPython.display.html#IPython.display.Markdown) renders the headings and links in the notebook.

To list the sources, [`re.findall`](https://docs.python.org/3/library/re.html#re.findall) extracts every link, i.e., a title in square brackets followed by a URL in parentheses. [`drop_duplicates`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop_duplicates.html) keeps each URL once, and [`pd.option_context`](https://pandas.pydata.org/docs/reference/api/pandas.option_context.html) temporarily disables the truncation of long cells so that URLs are shown in full.

In [11]:
links = re.findall(r"\[([^\]]+)\]\((https?://[^)\s]+)\)", text)
sources = pd.DataFrame(links, columns=["title", "url"]).drop_duplicates("url", ignore_index=True)
with pd.option_context("display.max_colwidth", None):
    display(sources)

,title,url
0,Wikipedia - 2024 Spanish floods,https://en.wikipedia.org/wiki/2024_Spanish_floods
1,WMO,https://wmo.int/media/news/devastating-rainfall-hits-spain-yet-another-flood-related-disaster
2,BBC News,https://www.bbc.com/news/articles/cx2dp775nd1o
3,El País English,https://english.elpais.com/climate/2025-04-15/70-of-flood-deaths-in-europe-in-2024-occurred-in-the-valencia-floodplain.html
4,The Olive Press,https://www.theolivepress.es/spain-news/2026/09/29/valencia-2024-flood-disaster-death-toll-goes-up-to-233-people/
5,Guy Carpenter Post-Event Report,https://www.guycarp.com/insights/2024/11/october-2024-dana-floods-spain.html
6,PreventionWeb,https://www.preventionweb.net/news/study-concludes-climate-change-intensified-flooding-caused-dana-valencia-2024
7,Euro Weekly News,https://euroweeklynews.com/2025/10/28/one-year-after-the-dana-floods-229-lives-lost-billions-in-damages-and-a-region-still-in-shock/
8,JBA Risk Management,https://www.jbarisk.com/knowledge-hub/event-response/spain-floods-october-2024/
9,International Journal for Equity in Health,https://equityhealthj.biomedcentral.com/articles/10.1186/s12939-025-02435-0


## Use the Storylines of Ronco et al. (2026)

Before generating storylines for many events, check whether Ronco et al. already provide them. Their dataset is a single CSV file, `DisasterStory.csv`, published on [Zenodo](https://doi.org/10.5281/zenodo.18598183) and also browsable in an [online dashboard](https://huggingface.co/spaces/roncmic/crisesStorylinesRAG).

### Load the Dataset

[`pd.read_csv`](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) reads the file directly from its URL. As for EM-DAT, we use `DisNo.` as the index.

In [12]:
RONCO_URL = "https://zenodo.org/records/18598183/files/DisasterStory.csv?download=1"
ronco = pd.read_csv(RONCO_URL, index_col="DisNo.")
print(f"Number of storylines: {len(ronco)}")
ronco.columns

Number of storylines: 3158


Index(['Disaster Type', 'Country', 'Locations', 'nNews', 'start_dt',
       'key information', 'severity', 'key drivers',
       'main impacts, exposure, and vulnerability',
       'likelihood of multi-hazard risks',
       'best practices for managing this risk',
       'recommendations and supportive measures for recovery', 'llama graph'],
      dtype='str')

Each row is one EM-DAT event, identified by its `DisNo.`. The columns contain:

- a few event details: `Disaster Type`, `Country`, `Locations`, the start date `start_dt`, and `nNews`, the number of news articles used;
- the seven storyline sections, in lowercase;
- `llama graph`, the knowledge graph extracted from the storyline, as a list of *(subject, predicate, object)* triples.

### Check the Coverage

The storylines cover events from 2014 to 2024. Let us count how many EM-DAT events of this period have one: [`between`](https://pandas.pydata.org/docs/reference/api/pandas.Series.between.html) selects the years, and [`isin`](https://pandas.pydata.org/docs/reference/api/pandas.Index.isin.html) tells, for each event, whether its `DisNo.` appears in the dataset of Ronco et al.

In [13]:
emdat = load_emdat()
recent = emdat[emdat["Start Year"].between(2014, 2024)]
has_storyline = recent.index.isin(ronco.index)
print(f"{has_storyline.sum()} of {len(recent)} EM-DAT events have a storyline ({has_storyline.mean():.0%})")

3154 of 6244 EM-DAT events have a storyline (51%)


About half of the events are covered. The count is slightly below the number of storylines because a few storylines refer to events absent from this version of the EM-DAT Archive.

`has_storyline` contains `True`/`False` values, so its mean gives the share of events covered. Let us compute it per disaster type: [`assign`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.assign.html) adds `has_storyline` as a column, and [`groupby`](https://pandas.pydata.org/docs/user_guide/groupby.html) averages it within each type.

In [14]:
coverage = (
    recent.assign(has_storyline=has_storyline)
    .groupby("Disaster Type")["has_storyline"]
    .mean()
)
coverage[["Earthquake", "Wildfire", "Storm", "Flood", "Drought", "Extreme temperature"]].round(2)

Disaster Type
Earthquake             0.84
Wildfire               0.83
Storm                  0.56
Flood                  0.54
Drought                0.18
Extreme temperature    0.14
Name: has_storyline, dtype: float64

Coverage is uneven: sudden-onset disasters, such as earthquakes and wildfires, are well covered, whereas slow-onset disasters, such as droughts and extreme temperatures, rarely are, likely because they receive less media attention.

### Add Storylines to EM-DAT Records

Since both tables share the same index, [`join`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.join.html) adds storyline sections to any selection of EM-DAT records. Let us try it on all events recorded in Spain in 2024, selected with a [boolean mask](https://pandas.pydata.org/docs/user_guide/indexing.html#boolean-indexing):

In [15]:
spain = recent[(recent["Country"] == "Spain") & (recent["Start Year"] == 2024)]
augmented = spain[["Disaster Type"]].join(ronco[["key information", "nNews"]])
augmented

,Disaster Type,key information,nNews
DisNo.,,,
2024-9269-ESP,Drought,NaN,NaN
2024-0086-ESP,Fire (Miscellaneous),"A fire occurred in a tower block in Valencia, ...",2.0
2024-0129-ESP,Storm,NaN,NaN
2024-0369-ESP,Extreme temperature,NaN,NaN
2024-0796-ESP,Flood,"A severe flood disaster occurred in Spain, aff...",22.0
2024-0839-ESP,Flood,"A flood disaster occurred in Malaga, Spain, sp...",2.0
2024-0853-ESP,Fire (Miscellaneous),NaN,NaN
2024-0949-ESP,Water,NaN,NaN


Only three events have a storyline; the others get missing values (`NaN`). As expected from the coverage above, they include the drought and the heatwave. Note also that two storylines rely on only two news articles (`nNews`): read such storylines with extra caution.

[`isna`](https://pandas.pydata.org/docs/reference/api/pandas.Series.isna.html) selects the events without a storyline, which an agent such as ours could document:

In [16]:
to_document = augmented.index[augmented["key information"].isna()]
list(to_document)

['2024-9269-ESP',
 '2024-0129-ESP',
 '2024-0369-ESP',
 '2024-0853-ESP',
 '2024-0949-ESP']

### Compare with the Agent

Let us display the storyline of the Valencia floods by Ronco et al., using the four sections of our template:

In [17]:
disno = "2024-0796-ESP"
for section in SECTIONS:
    content = ronco.loc[disno, section.lower()]  # column names are in lowercase
    display(Markdown(f"#### {section}\n\n{content}"))

#### Key information

A severe flood disaster occurred in Spain, affecting various municipalities in Valencia, including Alaquàs, Albal, Aldaia, Alfafar, Algemesí, Benetússer, Catarroja, Llocnou de la Corona, Massanassa, Paiporta, Picanya, Sedaví, and others, as well as Letur in Albacete province, and other regions such as Castilla-La Mancha, Andalusia, Murcia, the Balearic Islands, and Catalonia, on October 29, 2024

#### Severity

High

#### Key drivers

The main cause of the disaster was a DANA (a type of severe weather phenomenon) that brought heavy rainfall to the region, with up to a year's worth of rain falling in a few hours

#### Main impacts, exposure, and vulnerability

Economic damage: Unknown; People affected: Thousands, with many displaced and in need of aid; Fatalities: At least 220 people have died, with 8 still missing; Effects on communities and infrastructure: Severe damage to roads, homes, businesses, and infrastructure, with many areas left without water or power, and significant disruption to transportation and daily life

And here are the impact figures recorded in EM-DAT:

In [18]:
load_by_disno(disno)[["Total Deaths", "Total Affected", "Total Damage ('000 US$)"]]

Total Deaths                    232.0
Total Affected                36115.0
Total Damage ('000 US$)    11000000.0
Name: 2024-0796-ESP, dtype: object

Comparing these with the agent's storyline above, a few observations stand out:

- **The evidence differs.** Ronco et al. used news published within four weeks of the event onset, from a curated archive. The agent searches the web as it is today, including reports published months after the event.
- **Figures change over time.** Within four weeks, the news reported at least 220 deaths and 8 missing people, close to EM-DAT's 232, a total that combines deceased and missing people (see [human impact variables](https://doc.emdat.be/docs/data-structure-and-content/impact-variables/human/)). The economic damage was still unknown at the time, whereas EM-DAT now records USD 11 billion. Check which figures the agent found, and from which sources.
- **Traceability.** The agent's statements point to linked sources, which makes them easier to verify, whereas the dataset of Ronco et al. does not list the articles used.

## Retrieve Structured Outputs

Storylines are written for humans. To process them with code, e.g., to select events by severity, store impact figures, or write standard summaries, we need structured data. This is what [structured outputs](https://docs.mistral.ai/studio/conversations/structured-output) are for: the model answers in JSON, following a schema that we define.

### Define a Schema

The storyline template of Ronco et al. asks for a severity level (low, medium, or high) without defining these levels. The model then decides for itself, and the levels say little. In the dataset of Ronco et al., for instance, most storylines rate the severity of their event as high:

In [19]:
share_high = ronco["severity"].str.lower().str.startswith("high").mean()
print(f"Share of storylines rated 'high': {share_high:.0%}")

Share of storylines rated 'high': 87%


We therefore define the levels in our schema, written with [Pydantic](https://pydantic.dev/docs/validation/latest/concepts/models/), a data validation library. A schema is a class that lists fields and their types:

- [`Literal`](https://docs.python.org/3/library/typing.html#typing.Literal) restricts the severity to three values;
- `int | None` allows a missing value, so that the model does not have to invent a figure;
- a field can itself be a schema: `Range` stores the lowest and highest figures reported, as figures often vary between sources, and `Damage` extends it with a currency;
- the descriptions given with [`Field`](https://pydantic.dev/docs/validation/latest/concepts/fields/) are sent to the model: they are part of the instructions.

In [20]:
class Range(BaseModel):
    min: int | None = Field(description="Lowest figure reported, or the only one")
    max: int | None = Field(description="Highest figure reported, or the only one")


class Damage(Range):
    currency: str | None = Field(description="Currency of the figures, e.g., USD or EUR")


class Impacts(BaseModel):
    severity: Literal["low", "medium", "high"] = Field(
        description="low: fewer than 10 deaths and local impacts; "
        "medium: 10 to 99 deaths, or impacts on a whole region; "
        "high: 100 deaths or more, or impacts at the national scale"
    )
    deaths: Range = Field(description="Number of deaths")
    missing: Range = Field(description="Number of missing people")
    affected: Range = Field(description="Number of people affected")
    damage: Damage = Field(description="Economic damage, in millions, in the currency reported")

The severity thresholds are only an example: adapt them to your needs. Note also that the schema asks for figures as reported, e.g., deaths and missing people separately, and damage in its original currency. Adding up figures or converting currencies is more reliable when done by you than by the model.

### Extract the Values

[`client.chat.parse`](https://docs.mistral.ai/studio/conversations/structured-output/custom) sends a storyline to the model along with the schema, and converts the answer into an `Impacts` object. We do not need the agent here, as all the information is in the text. We keep the model and temperature of the agent, and add `reasoning_effort="none"` to turn off [reasoning](https://docs.mistral.ai/studio/conversations/reasoning): this simple task does not need it, and each call takes a few seconds instead of about a minute.

In [21]:
EXTRACTION_PROMPT = (
    "Extract the severity and impacts of the disaster described in the storyline. "
    "Rate the severity with the levels defined in the schema, not with the severity stated in the storyline. "
    "Use null for any figure that is not reported."
)


def extract_impacts(storyline: str) -> Impacts:
    response = client.chat.parse(
        model="mistral-large-4",
        messages=[
            {"role": "system", "content": EXTRACTION_PROMPT},
            {"role": "user", "content": storyline},
        ],
        response_format=Impacts,
        temperature=0,
        top_p=1,
        reasoning_effort="none",
    )
    return response.choices[0].message.parsed


impacts = extract_impacts(text)
impacts

Impacts(severity='high', deaths=Range(min=229, max=233), missing=Range(min=None, max=None), affected=Range(min=300000, max=300000), damage=Damage(min=10700, max=29000, currency='EUR'))

Each field is an attribute of the result, e.g., `impacts.deaths.max`.

**Note**: Agents also accept a `response_format` in their `completion_args`, so our agent could return JSON directly. We keep two steps: the storyline remains readable, and `extract_impacts` works on any text, such as the storylines of Ronco et al.

### Convert to JSON

[JSON](https://www.json.org/json-en.html) is a common text format to store and exchange data. [`model_dump_json`](https://pydantic.dev/docs/validation/latest/concepts/serialization/#modelmodel_dump_json) converts the result into a JSON string, which you can save to a file, e.g., with [`Path.write_text`](https://docs.python.org/3/library/pathlib.html#pathlib.Path.write_text):

In [22]:
impacts_json = impacts.model_dump_json(indent=2)
print(impacts_json)

{
  "severity": "high",
  "deaths": {
    "min": 229,
    "max": 233
  },
  "missing": {
    "min": null,
    "max": null
  },
  "affected": {
    "min": 300000,
    "max": 300000
  },
  "damage": {
    "min": 10700,
    "max": 29000,
    "currency": "EUR"
  }
}


Conversely, [`model_validate_json`](https://pydantic.dev/docs/validation/latest/concepts/json/#json-parsing) reads a JSON string back into an `Impacts` object, and raises an error if it does not match the schema, e.g., if the severity is not one of the three levels:

In [23]:
Impacts.model_validate_json(impacts_json)

Impacts(severity='high', deaths=Range(min=229, max=233), missing=Range(min=None, max=None), affected=Range(min=300000, max=300000), damage=Damage(min=10700, max=29000, currency='EUR'))

### Write a Summary

Structured outputs also make it easy to write standard summaries. Rather than asking the model, we fill a text template with values from the EM-DAT record and from the extraction: every summary then follows the same format and contains only values that we can check.

The function `format_range` writes a range as text, using the [format specifier](https://docs.python.org/3/library/string.html#format-specification-mini-language) `:,` to separate thousands. Then, [`str.format`](https://docs.python.org/3/library/stdtypes.html#str.format) replaces each `{placeholder}` of the template with a value.

In [24]:
SUMMARY_TEMPLATE = """#### {disaster_type} in {country} ({disno})

**Severity**: {severity}

- Deaths: {deaths}
- Missing people: {missing}
- People affected: {affected}
- Economic damage: {damage}"""


def format_range(values: Range, unit: str = "") -> str:
    """Write a range as text, e.g., '224 to 232', 'at least 220', or 'unknown'."""
    low, high = values.min, values.max
    if low is None and high is None:
        return "unknown"
    if high is None:
        formatted = f"at least {low:,}"
    elif low is None:
        formatted = f"up to {high:,}"
    elif low == high:
        formatted = f"{low:,}"
    else:
        formatted = f"{low:,} to {high:,}"
    return formatted + unit


summary = SUMMARY_TEMPLATE.format(
    disaster_type=disaster["Disaster Type"],
    country=disaster["Country"],
    disno=disaster.name,  # the index label of the record, i.e., its DisNo.
    severity=impacts.severity,
    deaths=format_range(impacts.deaths),
    missing=format_range(impacts.missing),
    affected=format_range(impacts.affected),
    damage=format_range(impacts.damage, unit=f" million {impacts.damage.currency}"),
)
display(Markdown(summary))

#### Flood in Spain (2024-0796-ESP)

**Severity**: high

- Deaths: 229 to 233
- Missing people: unknown
- People affected: 300,000
- Economic damage: 10,700 to 29,000 million EUR

To summarise several events, call `extract_impacts` and fill the template in a loop, e.g., over the storylines of Ronco et al.

## Good Practices and Limitations

- **Check the sources.** Citations make a storyline verifiable, not necessarily correct: open the links and confirm the key facts. Likewise, compare extracted values with the storyline.
- **Treat figures as leads, not data.** Numbers in the news are often preliminary and may follow other definitions than EM-DAT's, e.g., counting confirmed deaths only. EM-DAT figures follow documented [protocols](https://doc.emdat.be/docs/protocols/) and remain the reference.
- **Favour reproducibility.** For research, use a named model version and a temperature of 0, as done here. Answers can still vary, if only because web search results change over time, so save the storylines you generate together with their sources.
- **Mind the biases.** Like news archives, web search favours well-covered events reported in English: slow-onset disasters and events in regions with less media attention are harder to document.
- **Estimate costs before scaling up.** Each storyline requires one or more web searches, and the pages found are passed to the model, which consumes many tokens (see `response.usage`). Test the agent on a few events before running it on hundreds.

Finally, let us delete the agent, as we no longer need it:

In [25]:
client.beta.agents.delete(agent_id=agent.id)

## Conclusion

In this tutorial, we used a Mistral agent with web search to turn an EM-DAT record into a storyline with cited sources, and compared it with the dataset of Ronco et al. (2026), which covers about half of the EM-DAT events from 2014 to 2024. With structured outputs, we then extracted a severity level and impact figures from the storyline, converted them to JSON, and wrote a standard summary.

In practice, start from their dataset, and use an agent for the events it does not cover or that occurred after 2024. To process several events, call the agent in a loop over their `DisNo.`. By changing the instructions or the schema, the same approach can serve other purposes, e.g., extracting the affected locations of an event.

Ronco et al. also turned each storyline into a knowledge graph of *causes* and *prevents* relationships with a second LLM call; their code is available on [Zenodo](https://doi.org/10.5281/zenodo.18598183).

## References

- Charalampous, P., Speybroeck, N., van Loenhout, J. A. F., Pluen, G., & Delforge, D. (2025). The 2024 Spain floods: A call for resilience and the duty of memory. *International Journal of Public Health*, 70, 1608236. https://doi.org/10.3389/ijph.2025.1608236
- Delforge, D., Wathelet, V., Below, R., Lanfredi Sofia, C., Tonnelier, M., van Loenhout, J. A. F., & Speybroeck, N. (2024). *The EM-DAT Emergency Events Database Archive* (Version 2) [Data set]. Open Data @ UCLouvain. https://doi.org/10.14428/DVN/I0LTPH
- Delforge, D., et al. (2025). EM-DAT: The Emergency Events Database. *International Journal of Disaster Risk Reduction*, 124, 105509. https://doi.org/10.1016/j.ijdrr.2025.105509
- Ronco, M., Bandelli, L., Bertolini, L., Consoli, S., Delforge, D., Spadaro, A., Verile, M., & Corbane, C. (2026). Disaster storylines and knowledge graphs from global news with large language models and retrieval-augmented generation. *Scientific Data*, 13, 689. https://doi.org/10.1038/s41597-026-07036-2
- Ronco, M., et al. (2026). *crisesStorylinesRAG* [Data set]. Zenodo. https://doi.org/10.5281/zenodo.18598183